In [1]:
from dotenv import load_dotenv
import json
from uuid import uuid4
from pathlib import Path
from datetime import datetime, timezone
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader

load_dotenv()

C:\2026-Projects\Document_Intelligent_Hub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\benne\AppData\Local\Temp\ipykernel_9900\2927043132.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


True

In [2]:
llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

In [3]:
PROJECT_ROOT = Path.cwd()

RAW_DATA_DIR= PROJECT_ROOT / "data"/"raw"
PROCESSED_DATA_DIR= PROJECT_ROOT / "data"/ "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Raw data directory: C:\document_hub\notebooks\data\raw
Processed data directory: C:\document_hub\notebooks\data\processed


In [4]:
files = list(RAW_DATA_DIR.glob("*"))

for file in files:
    print(file.name)

Sample Compliance Manual.pdf


In [5]:

pdf_files= list(RAW_DATA_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDF files found.")
else:
    pdf_path = pdf_files[0]
    loader = PyPDFLoader(str(pdf_path))
    pdf_documents = loader.load()

    print(f"Loaded: {pdf_path.name}")
    print(f"Pages extracted: {len(pdf_documents)}")
    print()
    print(pdf_documents[0].page_content[:1000])

Loaded: Sample Compliance Manual.pdf
Pages extracted: 3

Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere to these standards may result in 
disciplinary action, financial penalties, or regulatory sanctions. 
The manual is designed to: 
• Provide clear obligations for staff. 
• Establish reporting thresholds and escalation procedures. 
• Ensure alignment with national and international regulatory frameworks. 
 
Page 2 — Anti-Money Laundering (AML) Obligations 
Transaction Monitoring 
• All transactions exceeding USD 10,000 must be reported to the Compliance 
Office within 24 hours. 
• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious 

In [6]:
def enrich_document_metadata(
        documents: list[Document],
        file_path: Path,
        department: str = "Compliance",
        access_role: str = "Compliance Analyst",
) -> list[Document]:
    document_id= str(uuid4())
    ingested_at= datetime.now(timezone.utc).isoformat()

    enriched_documents = []

    for index, document in enumerate(documents):
        original_page = document.metadata.get("page")

        page_number = None
        if original_page is not None:
            page_number = int(original_page) +1

        document.metadata.update(
            {
                "document_id": document_id,
                "source": file_path.name,
                "file_path": str(file_path),
                "file_type": file_path.suffix.lower().replace(".", ""),
                "department": department,
                "access_role": access_role,
                "ingested_at": ingested_at,
                "page_number": page_number,
                "part_index": index,
            }

        )

        enriched_documents.append(document)
    return enriched_documents

In [7]:
def load_single_file(file_path: Path) -> list[Document]:
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        loader = PyPDFLoader(str(file_path))
        documents = loader.load()

    elif suffix == ".txt":
        loader = TextLoader(str(file_path), encoding="utf-8")
        documents = loader.load()

    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    return enrich_document_metadata(
        documents=documents,
        file_path= file_path
    )

In [8]:
def ingest_folder(folder_path: Path) -> list[Document]:
    supported_extensions = {".pdf", ".txt"}

    all_documents = [ ]

    for file_path in folder_path.iterdir():
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in supported_extensions:
            print(f"Skipping {file_path.name} as it is not a supported file type.")
            continue

        try:
            documents = load_single_file(file_path)
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {file_path.name}")

        except Exception as error:
            print(f"Failed to load {file_path.name}: {error}")

    return all_documents


In [9]:
def documents_to_json_records(documents: list[Document]) -> list[dict]:
    records = []
    for document in documents:
        records.append(
            {
                "page_content": document.page_content,
                "metadata": document.metadata,
            }
        )
    return records

In [10]:
def json_records_to_documents(records:list[dict]) -> list[Document]:
    documents =[]

    for record in records:
        documents.append(
            Document(
                page_content=record["page_content"],
                metadata=record["metadata"]
            )
        )

    return documents


In [11]:
ingested_documents = ingest_folder(RAW_DATA_DIR)
print(f"Total documents ingested: {len(ingested_documents)}")

Loaded 3 documents from Sample Compliance Manual.pdf
Total documents ingested: 3


In [12]:
if ingested_documents:
    first_doc = ingested_documents[0]

    print("Metadata:")
    print(first_doc.metadata)

    print()
    print("Content preview:")
    print(first_doc.page_content[:1000])



Metadata:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': 'a7777030-0f2d-4a90-a97e-c92105894b71', 'file_path': 'C:\\document_hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-26T16:46:03.274307+00:00', 'page_number': 1, 'part_index': 0}

Content preview:
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere t

In [13]:
output_path = PROCESSED_DATA_DIR / "ingested_documents.json"

records = documents_to_json_records(ingested_documents)

with output_path.open("w", encoding="utf-8") as file:
    json.dump(records, file, indent=2, ensure_ascii=False)

print(f"Saved {len(records)} records to {output_path}")

with output_path.open("r", encoding="utf-8") as file:
    loaded_records = json.load(file)

reloaded_documents = json_records_to_documents(loaded_records)

print(f"Reloaded {len(reloaded_documents)} documents.")

if reloaded_documents:
    print(reloaded_documents[0].metadata)

Saved 3 records to C:\document_hub\notebooks\data\processed\ingested_documents.json
Reloaded 3 documents.
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': 'a7777030-0f2d-4a90-a97e-c92105894b71', 'file_path': 'C:\\document_hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-26T16:46:03.274307+00:00', 'page_number': 1, 'part_index': 0}
